# Notebook 13 (V4_3) - Approximation Error in Terminal Tails

This notebook implements `AI_gen_prompts/Approximation_error_in_tails_analysis.md` for the two-country production economy, using the large-bubble `12_v9` calibration:

| object | value |
|---|---:|
| `gamma` | 0.25 |
| `pi_persist` | 0.80 |
| `xi_u` | 2.00 |
| `nu_u` | 1.50 |
| seed `nu_b`, `xi_W` | 0.50 |
| `common_world_growth` | true |
| `branch_iters` | 100 |

Two checks are run:

1. Horizon/buffer invariance around the certified `12_v9` baseline (`T_report = 45`, `n_buffer = 5`).
2. Terminal-rule sensitivity under four artificial successor rules: flat, local log-linear, asymptotic HKT-rate, and tail regression.

The default grid is intentionally smaller than the original prompt grid but still replication-grade. For a quick syntax/execution check, set `RUN_MODE = :smoke` in the first code cell, or run with `TAIL_ANALYSIS_MODE=smoke`.

In [ ]:
using Pkg

function find_project_dir(start_dir=pwd())
    dir = start_dir
    while true
        isfile(joinpath(dir, "Project.toml")) && return dir
        parent = dirname(dir)
        parent == dir && return start_dir
        dir = parent
    end
end

const PROJECT_DIR = find_project_dir()
Pkg.activate(PROJECT_DIR)

include(joinpath(PROJECT_DIR, "Two_country_production", "TwoCountryProductionOLG.jl"))

using Printf, Statistics, Markdown
using Plots, LaTeXStrings
using Plots.PlotMeasures

gr()
default(size=(940, 560), framestyle=:box, grid=:y, legend=:best,
        fontfamily="Computer Modern", linewidth=2,
        titlefontsize=11, guidefontsize=10, tickfontsize=9, legendfontsize=8,
        left_margin=8mm, right_margin=6mm, top_margin=7mm, bottom_margin=8mm)

const OUTDIR = joinpath(PROJECT_DIR, "Two_country_production", "outputs_v43",
                        "approximation_error_in_tails")
isdir(OUTDIR) || mkpath(OUTDIR)

RUN_MODE = Symbol(get(ENV, "TAIL_ANALYSIS_MODE", "replication"))
@assert RUN_MODE in (:replication, :smoke) "RUN_MODE must be :replication or :smoke"

println("Project directory: ", PROJECT_DIR)
println("Output directory:  ", OUTDIR)
println("Run mode:          ", RUN_MODE)

## 0. Calibration and Grid

Replication mode uses the agreed smaller grid:

- Horizon/buffer grid: `T_report in {45, 60}`, `n_buffer in {0, 5, 20}`.
- Baseline reference: `T_report = 45`, `n_buffer = 5`, matching the certified `12_v9` common-growth run.
- Comparison windows: `T0 in {20, 30, 40}`.
- Terminal-rule sensitivity: `T_report = 45`, `n_buffer in {0, 5, 20}`.

Smoke mode keeps the same economic calibration but uses tiny horizons so the notebook can be checked quickly.

In [ ]:
const RESID_TOL = 1e-5

function tail_combo_params(; T_max::Int, n_buffer::Int)
    return ProductionParams(T_max=T_max,
        γ=0.25,
        π_persist=0.80,
        ξ_u=2.0,
        ν_u=1.5,
        ν_b=0.5,
        ξ_W=0.5,
        common_world_growth=true,
        branch_iters=100,
        n_buffer=n_buffer,
        do_global_polish=false)
end

if RUN_MODE == :smoke
    BASELINE_SPEC = (T_report=5, n_buffer=2)
    HORIZON_T_GRID = [5, 6]
    BUFFER_GRID = [0, 2]
    T0_GRID = [2, 4]
    TERMINAL_T_REPORT = 5
    TERMINAL_BUFFER_GRID = [0, 2]
    TAIL_REGRESSION_K_DEFAULT = 4
else
    BASELINE_SPEC = (T_report=45, n_buffer=5)
    HORIZON_T_GRID = [45, 60]
    BUFFER_GRID = [0, 5, 20]
    T0_GRID = [20, 30, 40]
    TERMINAL_T_REPORT = 45
    TERMINAL_BUFFER_GRID = [0, 5, 20]
    TAIL_REGRESSION_K_DEFAULT = 8
end

TERMINAL_RULES = [:flat, :local_loglinear, :asymptotic_hkt, :tail_regression]

println("Baseline spec: ", BASELINE_SPEC)
println("Horizon grid:  T_report=", HORIZON_T_GRID, ", n_buffer=", BUFFER_GRID)
println("T0 windows:    ", T0_GRID)
println("Terminal grid: T_report=", TERMINAL_T_REPORT, ", n_buffer=", TERMINAL_BUFFER_GRID)

## 1. Notebook-Local Terminal Closure Dispatcher

The production module is left untouched. After `include(...)`, this cell replaces the existing `_extrapolate_terminal!` method with a dispatcher controlled by `TAIL_TERMINAL_RULE[]`.

The current model rule is `:local_loglinear`, i.e. `phi_{T+1} = phi_T^2 / phi_{T-1}` for both country labour-allocation coordinates. The HKT asymptotic-rate rule is applied to the U.S. unbalanced branch; the RoW coordinate is carried by the local log-linear rule because the HKT decay bound is a U.S. unbalanced-branch condition.

In [ ]:
const TAIL_TERMINAL_RULE = Ref(:local_loglinear)
const TAIL_TERMINAL_PARAMS = Ref{Any}(nothing)
const TAIL_REGRESSION_K = Ref(TAIL_REGRESSION_K_DEFAULT)

phi_clamp(x) = clamp(x, 1e-8, 1 - 1e-8)

function local_loglinear_phi(pol::Matrix{Float64}, i::Int, T::Int)
    if T >= 2
        a, b = pol[i, T-1], pol[i, T]
        if isfinite(a) && isfinite(b) && a > 1e-12 && b > 1e-12
            return phi_clamp(b * b / a)
        end
    end
    return phi_clamp(pol[i, T])
end

function tail_regression_phi(pol::Matrix{Float64}, i::Int, T::Int; k::Int=8)
    lo = max(1, T - k + 1)
    idx = [t for t in lo:T if isfinite(pol[i, t]) && pol[i, t] > 1e-12]
    length(idx) < 3 && return local_loglinear_phi(pol, i, T)

    x = Float64.(idx)
    y = log.([pol[i, t] for t in idx])
    xbar = mean(x); ybar = mean(y)
    sxx = sum(abs2, x .- xbar)
    sxx <= 0 && return local_loglinear_phi(pol, i, T)

    slope = sum((x .- xbar) .* (y .- ybar)) / sxx
    intercept = ybar - slope * xbar
    return phi_clamp(exp(intercept + slope * (T + 1)))
end

function asymptotic_hkt_phi_us(pol::Matrix{Float64}, T::Int)
    p = TAIL_TERMINAL_PARAMS[]
    p === nothing && return local_loglinear_phi(pol, 1, T)

    phi_T = phi_clamp(pol[1, T])
    psi_US = (p.ξ_u - p.ν_u) * (p.ρ_US - 1)
    asymptotic_rate = G_N_US(p, phi_T)^(-psi_US / p.ρ_US)
    if isfinite(asymptotic_rate) && asymptotic_rate > 0
        return phi_clamp(phi_T * asymptotic_rate)
    end
    return local_loglinear_phi(pol, 1, T)
end

# Notebook-local replacement of the model's terminal closure.
function _extrapolate_terminal!(pol::Matrix{Float64}, T::Int)
    pol[:, T+1] .= pol[:, T]
    rule = TAIL_TERMINAL_RULE[]

    if rule == :flat
        return pol
    elseif rule == :local_loglinear
        pol[1, T+1] = local_loglinear_phi(pol, 1, T)
        pol[2, T+1] = local_loglinear_phi(pol, 2, T)
    elseif rule == :asymptotic_hkt
        pol[1, T+1] = asymptotic_hkt_phi_us(pol, T)
        pol[2, T+1] = local_loglinear_phi(pol, 2, T)
    elseif rule == :tail_regression
        pol[1, T+1] = tail_regression_phi(pol, 1, T; k=TAIL_REGRESSION_K[])
        pol[2, T+1] = tail_regression_phi(pol, 2, T; k=TAIL_REGRESSION_K[])
    else
        error("Unknown terminal rule: $rule")
    end
    return pol
end

function set_terminal_rule!(rule::Symbol, p::ProductionParams)
    @assert rule in TERMINAL_RULES "unknown terminal rule"
    TAIL_TERMINAL_RULE[] = rule
    TAIL_TERMINAL_PARAMS[] = p
    return rule
end

println("Installed notebook-local terminal closure dispatcher.")

## 2. Helpers

Each solve returns the equilibrium path, the fundamental-value recursion, a compact certification row, and any caught error. Results are cached by `(T_report, n_buffer, terminal_rule)` so the local-loglinear runs used in both sections are not repeated.

In [ ]:
path_vector(result, field::Symbol) = [getfield(s, field) for s in result.u_path]

function positive_for_log(x; floor=1e-20)
    return max.(Float64.(x), floor)
end

function pretty(x)
    x === missing && return ""
    x isa Bool && return string(x)
    x isa Integer && return string(x)
    if x isa AbstractFloat
        if !isfinite(x)
            return string(x)
        elseif abs(x) >= 1000 || (abs(x) > 0 && abs(x) < 1e-3)
            return @sprintf("%.3e", x)
        else
            return @sprintf("%.6f", x)
        end
    end
    return string(x)
end

function html_escape(x)
    s = string(x)
    s = replace(s, "&" => "&amp;")
    s = replace(s, "<" => "&lt;")
    s = replace(s, ">" => "&gt;")
    s = replace(s, "_" => "\\_")
    return s
end

function markdown_table(rows, cols; maxrows=length(rows))
    isempty(rows) && return Markdown.parse("_No rows._")
    io = IOBuffer()
    println(io, "| ", join([html_escape(c) for c in cols], " | "), " |")
    println(io, "| ", join(fill("---", length(cols)), " | "), " |")
    for r in rows[1:min(maxrows, length(rows))]
        println(io, "| ", join([html_escape(pretty(getproperty(r, Symbol(c)))) for c in cols], " | "), " |")
    end
    Markdown.parse(String(take!(io)))
end

function csv_escape(x)
    x === missing && return ""
    s = string(x)
    if occursin(',', s) || occursin('"', s) || occursin('\n', s)
        return "\"" * replace(s, "\"" => "\"\"") * "\""
    end
    return s
end

function write_csv(filename, rows, cols)
    open(filename, "w") do io
        println(io, join(cols, ","))
        for r in rows
            println(io, join([csv_escape(getproperty(r, Symbol(c))) for c in cols], ","))
        end
    end
    return filename
end

const RESULT_CACHE = Dict{Tuple{Int,Int,Symbol},Any}()

function run_tail_case(T_report::Int, n_buffer::Int, rule::Symbol; verbose::Bool=false)
    key = (T_report, n_buffer, rule)
    haskey(RESULT_CACHE, key) && return RESULT_CACHE[key]

    p = tail_combo_params(T_max=T_report, n_buffer=n_buffer)
    set_terminal_rule!(rule, p)
    @printf("Solving T=%d, buffer=%d, terminal=%s\n", T_report, n_buffer, String(rule))

    t0 = time()
    try
        result = run_production_simulation(p; verbose=verbose)
        fv = fundamental_value_path(result)
        elapsed = time() - t0
        q = path_vector(result, :q_US)
        summary = (;
            T_report=T_report,
            n_buffer=n_buffer,
            terminal_rule=String(rule),
            elapsed_sec=elapsed,
            status="ok",
            branch_converged=result.branch_converged,
            max_u_residual=result.max_u_residual,
            max_bgp_residual=result.max_bgp_residual,
            psi_ok=result.diagnostics.psi_ok,
            psi_min=result.diagnostics.psi_min,
            equity_weights_ok=result.diagnostics.equity_weights_ok,
            equity_weight_min=result.diagnostics.equity_weight_min,
            nu_b_effective=result.params.ν_b,
            q_growth=q[end] / q[1],
            bubble_share_final=fv.bubble_share[end],
            bubble_share_max=maximum(fv.bubble_share),
            error="")
        out = (; status=:ok, result, fv, summary)
        RESULT_CACHE[key] = out
        return out
    catch err
        elapsed = time() - t0
        summary = (;
            T_report=T_report,
            n_buffer=n_buffer,
            terminal_rule=String(rule),
            elapsed_sec=elapsed,
            status="error",
            branch_converged=missing,
            max_u_residual=missing,
            max_bgp_residual=missing,
            psi_ok=missing,
            psi_min=missing,
            equity_weights_ok=missing,
            equity_weight_min=missing,
            nu_b_effective=missing,
            q_growth=missing,
            bubble_share_final=missing,
            bubble_share_max=missing,
            error=sprint(showerror, err))
        out = (; status=:error, result=nothing, fv=nothing, summary)
        RESULT_CACHE[key] = out
        @warn "Solve failed" T_report n_buffer rule err
        return out
    end
end

function comparison_metrics(case, ref, T0::Int)
    if case.status != :ok || ref.status != :ok
        return (max_log_q_error=missing,
                max_log_dividend_yield_error=missing,
                max_phi_error=missing,
                max_bubble_share_error=missing)
    end

    T = min(length(case.result.u_path), length(ref.result.u_path))
    if T0 > T
        return (max_log_q_error=missing,
                max_log_dividend_yield_error=missing,
                max_phi_error=missing,
                max_bubble_share_error=missing)
    end

    idx = 1:T0
    q = path_vector(case.result, :q_US)[idx]
    d = path_vector(case.result, :d_US)[idx]
    phi = path_vector(case.result, :φ_US)[idx]
    bshare = case.fv.bubble_share[idx]

    q_ref = path_vector(ref.result, :q_US)[idx]
    d_ref = path_vector(ref.result, :d_US)[idx]
    phi_ref = path_vector(ref.result, :φ_US)[idx]
    bshare_ref = ref.fv.bubble_share[idx]

    return (;
        max_log_q_error=maximum(abs.(log.(q) .- log.(q_ref))),
        max_log_dividend_yield_error=maximum(abs.(log.(d ./ q) .- log.(d_ref ./ q_ref))),
        max_phi_error=maximum(abs.(phi .- phi_ref)),
        max_bubble_share_error=maximum(abs.(bshare .- bshare_ref)))
end

## 3. Horizon and Buffer Invariance

Each candidate is compared with the `12_v9` baseline-centered reference using the current local log-linear terminal rule. The reported metrics match the prompt:

$$
\max_{t \le T_0}\left|\log q_t-\log q_t^{ref}\right|,
\quad
\max_{t \le T_0}\left|\log(d_t/q_t)-\log(d_t/q_t)^{ref}\right|,
$$
$$
\max_{t \le T_0}\left|\phi_t-\phi_t^{ref}\right|,
\quad
\max_{t \le T_0}\left|B_t/Q_t-(B_t/Q_t)^{ref}\right|.
$$

In [ ]:
baseline_case = run_tail_case(BASELINE_SPEC.T_report, BASELINE_SPEC.n_buffer, :local_loglinear)

horizon_cases = Any[]
for T_report in HORIZON_T_GRID, n_buffer in BUFFER_GRID
    push!(horizon_cases, run_tail_case(T_report, n_buffer, :local_loglinear))
end

horizon_case_summary = [c.summary for c in horizon_cases]

horizon_rows = NamedTuple[]
for c in horizon_cases
    for T0 in T0_GRID
        m = comparison_metrics(c, baseline_case, T0)
        push!(horizon_rows, merge((;
            T_report=c.summary.T_report,
            n_buffer=c.summary.n_buffer,
            terminal_rule=c.summary.terminal_rule,
            baseline_T_report=BASELINE_SPEC.T_report,
            baseline_n_buffer=BASELINE_SPEC.n_buffer,
            T0=T0,
            status=c.summary.status,
            max_u_residual=c.summary.max_u_residual,
            max_bgp_residual=c.summary.max_bgp_residual), m))
    end
end

markdown_table(horizon_rows,
    ["T_report", "n_buffer", "T0", "status", "max_u_residual",
     "max_log_q_error", "max_log_dividend_yield_error", "max_phi_error",
     "max_bubble_share_error"]; maxrows=length(horizon_rows))

In [ ]:
summary_cols = ["T_report", "n_buffer", "terminal_rule", "elapsed_sec", "status",
                "branch_converged", "max_u_residual", "max_bgp_residual", "psi_ok",
                "psi_min", "equity_weights_ok", "equity_weight_min", "nu_b_effective",
                "q_growth", "bubble_share_final", "bubble_share_max", "error"]
metric_cols = ["T_report", "n_buffer", "terminal_rule", "baseline_T_report",
               "baseline_n_buffer", "T0", "status", "max_u_residual",
               "max_bgp_residual", "max_log_q_error",
               "max_log_dividend_yield_error", "max_phi_error",
               "max_bubble_share_error"]

write_csv(joinpath(OUTDIR, "horizon_buffer_case_summary.csv"), horizon_case_summary, summary_cols)
write_csv(joinpath(OUTDIR, "horizon_buffer_invariance.csv"), horizon_rows, metric_cols)
println("Saved horizon/buffer tables to ", OUTDIR)

In [ ]:
plot_T0 = maximum(T0_GRID)
hrows_plot = filter(r -> r.T0 == plot_T0 && r.status == "ok" && r.max_log_q_error !== missing, horizon_rows)
labels = ["T=$(r.T_report), b=$(r.n_buffer)" for r in hrows_plot]
x = 1:length(hrows_plot)

p1 = plot(x, [r.max_log_q_error for r in hrows_plot], marker=:circle, lw=2.2,
          xticks=(x, labels), xrotation=35, ylabel="max error", title="Horizon/buffer errors through T0=$plot_T0",
          label=L"\log q")
plot!(p1, x, [r.max_log_dividend_yield_error for r in hrows_plot], marker=:square, lw=2.2,
      label=L"\log(d/q)")

p2 = plot(x, [r.max_phi_error for r in hrows_plot], marker=:circle, lw=2.2,
          xticks=(x, labels), xrotation=35, ylabel="max error", title="Policy and bubble-share errors",
          label=L"\varphi_{US}")
plot!(p2, x, [r.max_bubble_share_error for r in hrows_plot], marker=:square, lw=2.2,
      label=L"B/Q")

fig_horizon = plot(p1, p2, layout=(2,1), size=(980, 760), bottom_margin=12mm)
savefig(fig_horizon, joinpath(OUTDIR, "horizon_buffer_invariance_errors.png"))
fig_horizon

## 4. Terminal-Rule Sensitivity

This section compares four terminal successor rules at the `12_v9` calibration. For each buffer, every rule is compared with the local log-linear rule at the same reported horizon and buffer. This isolates closure sensitivity from the mechanical effect of changing the buffer length.

Rules:

- `flat`: `phi_{T+1} = phi_T`, intentionally bad near the terminal date.
- `local_loglinear`: current model rule, `phi_{T+1} = phi_T^2 / phi_{T-1}`.
- `asymptotic_hkt`: U.S. HKT decay-rate rule, with RoW local-loglinear.
- `tail_regression`: log-linear regression over the last `k` solved periods.

In [ ]:
terminal_cases = Any[]
for n_buffer in TERMINAL_BUFFER_GRID, rule in TERMINAL_RULES
    push!(terminal_cases, run_tail_case(TERMINAL_T_REPORT, n_buffer, rule))
end

terminal_case_summary = [c.summary for c in terminal_cases]

terminal_rows = NamedTuple[]
for n_buffer in TERMINAL_BUFFER_GRID
    ref = run_tail_case(TERMINAL_T_REPORT, n_buffer, :local_loglinear)
    for rule in TERMINAL_RULES
        c = run_tail_case(TERMINAL_T_REPORT, n_buffer, rule)
        for T0 in T0_GRID
            m = comparison_metrics(c, ref, T0)
            push!(terminal_rows, merge((;
                T_report=TERMINAL_T_REPORT,
                n_buffer=n_buffer,
                terminal_rule=String(rule),
                reference_rule="local_loglinear",
                T0=T0,
                status=c.summary.status,
                max_u_residual=c.summary.max_u_residual,
                max_bgp_residual=c.summary.max_bgp_residual), m))
        end
    end
end

markdown_table(terminal_rows,
    ["T_report", "n_buffer", "terminal_rule", "T0", "status", "max_u_residual",
     "max_log_q_error", "max_log_dividend_yield_error", "max_phi_error",
     "max_bubble_share_error"]; maxrows=length(terminal_rows))

In [ ]:
terminal_summary_cols = summary_cols
terminal_metric_cols = ["T_report", "n_buffer", "terminal_rule", "reference_rule",
                        "T0", "status", "max_u_residual", "max_bgp_residual",
                        "max_log_q_error", "max_log_dividend_yield_error",
                        "max_phi_error", "max_bubble_share_error"]

write_csv(joinpath(OUTDIR, "terminal_rule_case_summary.csv"), terminal_case_summary, terminal_summary_cols)
write_csv(joinpath(OUTDIR, "terminal_rule_sensitivity.csv"), terminal_rows, terminal_metric_cols)
println("Saved terminal-rule tables to ", OUTDIR)

In [ ]:
plot_buffer = BASELINE_SPEC.n_buffer in TERMINAL_BUFFER_GRID ? BASELINE_SPEC.n_buffer : first(TERMINAL_BUFFER_GRID)
plot_cases = [(rule, run_tail_case(TERMINAL_T_REPORT, plot_buffer, rule)) for rule in TERMINAL_RULES]
plot_cases = filter(rc -> rc[2].status == :ok, plot_cases)

p_q = plot(title="Terminal-rule sensitivity: q_US, buffer=$plot_buffer",
           xlabel="period t", ylabel=L"q_{US,t}", yscale=:log10, legend=:topleft)
p_phi = plot(title="Terminal-rule sensitivity: phi_US, buffer=$plot_buffer",
             xlabel="period t", ylabel=L"\varphi_{US,t}", legend=:bottomleft)
p_b = plot(title="Terminal-rule sensitivity: bubble share, buffer=$plot_buffer",
           xlabel="period t", ylabel=L"B/Q", legend=:topleft)

for (rule, c) in plot_cases
    tt = 1:length(c.result.u_path)
    plot!(p_q, tt, positive_for_log(path_vector(c.result, :q_US)), lw=2.2, label=String(rule))
    plot!(p_phi, tt, path_vector(c.result, :φ_US), lw=2.2, label=String(rule))
    plot!(p_b, tt, c.fv.bubble_share, lw=2.2, label=String(rule))
end

fig_terminal = plot(p_q, p_phi, p_b, layout=(3,1), size=(980, 980), bottom_margin=8mm)
savefig(fig_terminal, joinpath(OUTDIR, "terminal_rule_sensitivity_paths.png"))
fig_terminal

## 5. Reading the Evidence

For the horizon/buffer experiment, small errors through `T0 = 20, 30, 40` relative to the `12_v9` baseline indicate that reported early-path objects are not being driven by the artificial terminal successor.

For the terminal-rule experiment, the robust pattern to look for is:

- `local_loglinear`, `asymptotic_hkt`, and `tail_regression` remain close over pre-terminal windows.
- `flat` is the outlier, especially as `T0` approaches the terminal date or when `n_buffer = 0`.
- Adding a buffer should push any closure sensitivity away from the economically reported part of the path.

The CSVs and figures are written to `Two_country_production/outputs_v43/approximation_error_in_tails/`.